# WEEK 3 DAY 3

## Task 8: Create the notebook and connect

### Settings

In [0]:
dbutils.widgets.text("storage_account", "stbigdatadev00001")
dbutils.widgets.text("secret_scope", "kv-scope")

sa = dbutils.widgets.get("storage_account")
scope = dbutils.widgets.get("secret_scope")

spark.conf.set("spark.sql.session.timeZone", "UTC")
spark.conf.set(
    f"fs.azure.account.key.{sa}.dfs.core.windows.net",
    dbutils.secrets.get(scope, "storage-account-key"),
)

bronze_path = f"abfss://bronze@{sa}.dfs.core.windows.net/ecommerce/*/*/*.csv"
silver_path = f"abfss://silver@{sa}.dfs.core.windows.net/ecommerce_events"
rejected_path = f"abfss://silver@{sa}.dfs.core.windows.net/ecommerce_rejected"

### Check access

In [0]:
display(dbutils.fs.ls(f"abfss://bronze@{sa}.dfs.core.windows.net/ecommerce/"))

path,name,size,modificationTime
abfss://bronze@stbigdatadev00001.dfs.core.windows.net/ecommerce/year=2019/,year=2019/,0,1793783242000


## Task 9: Look at the raw data

### Read only a small sample first

In [0]:
from pyspark.sql import functions as F
from pyspark.sql.types import StructType, StructField, StringType

cols = ["event_time", "event_type", "product_id", "category_id", "category_code",
        "brand", "price", "user_id", "user_session"]
raw_schema = StructType([StructField(c, StringType(), True) for c in cols])

raw = spark.read.option("header", "true").schema(raw_schema).csv(bronze_path)
raw.printSchema()
display(raw.limit(10))

root
 |-- event_time: string (nullable = true)
 |-- event_type: string (nullable = true)
 |-- product_id: string (nullable = true)
 |-- category_id: string (nullable = true)
 |-- category_code: string (nullable = true)
 |-- brand: string (nullable = true)
 |-- price: string (nullable = true)
 |-- user_id: string (nullable = true)
 |-- user_session: string (nullable = true)



event_time,event_type,product_id,category_id,category_code,brand,price,user_id,user_session
2019-10-01 00:00:00 UTC,view,44600062,2103807459595387724,null,shiseido,35.79,541312140,72d76fde-8bb3-4e00-8c23-a032dfed738c
2019-10-01 00:00:00 UTC,view,3900821,2053013552326770905,appliances.environment.water_heater,aqua,33.20,554748717,9333dfbd-b87a-4708-9857-6336556b0fcc
2019-10-01 00:00:01 UTC,view,17200506,2053013559792632471,furniture.living_room.sofa,null,543.10,519107250,566511c2-e2e3-422b-b695-cf8e6e792ca8
2019-10-01 00:00:01 UTC,view,1307067,2053013558920217191,computers.notebook,lenovo,251.74,550050854,7c90fc70-0e80-4590-96f3-13c02c18c713
2019-10-01 00:00:04 UTC,view,1004237,2053013555631882655,electronics.smartphone,apple,1081.98,535871217,c6bd7419-2748-4c56-95b4-8cec9ff8b80d
2019-10-01 00:00:05 UTC,view,1480613,2053013561092866779,computers.desktop,pulser,908.62,512742880,0d0d91c2-c9c2-4e81-90a5-86594dec0db9
2019-10-01 00:00:08 UTC,view,17300353,2053013553853497655,null,creed,380.96,555447699,4fe811e9-91de-46da-90c3-bbd87ed3a65d
2019-10-01 00:00:08 UTC,view,31500053,2053013558031024687,null,luminarc,41.16,550978835,6280d577-25c8-4147-99a7-abc6048498d6
2019-10-01 00:00:10 UTC,view,28719074,2053013565480109009,apparel.shoes.keds,baden,102.71,520571932,ac1cd4e5-a3ce-4224-a2d7-ff660a105880
2019-10-01 00:00:11 UTC,view,1004545,2053013555631882655,electronics.smartphone,huawei,566.01,537918940,406c46ed-90a4-4787-a43b-59a410c1a5fb


## Task 10: Fix the types

In [0]:
typed = raw.select(
    F.to_timestamp("event_time", "yyyy-MM-dd HH:mm:ss 'UTC'").alias("event_time"),
    F.col("event_type"),
    F.col("product_id").cast("long"),
    F.col("category_id").cast("long"),
    F.col("category_code"),
    F.col("brand"),
    F.col("price").cast("decimal(10,2)"),
    F.col("user_id").cast("long"),
    F.col("user_session"),
)
typed.printSchema()
display(typed.limit(5))

root
 |-- event_time: timestamp (nullable = true)
 |-- event_type: string (nullable = true)
 |-- product_id: long (nullable = true)
 |-- category_id: long (nullable = true)
 |-- category_code: string (nullable = true)
 |-- brand: string (nullable = true)
 |-- price: decimal(10,2) (nullable = true)
 |-- user_id: long (nullable = true)
 |-- user_session: string (nullable = true)



event_time,event_type,product_id,category_id,category_code,brand,price,user_id,user_session
2019-10-01T00:00:00Z,view,44600062,2103807459595387724,null,shiseido,35.79,541312140,72d76fde-8bb3-4e00-8c23-a032dfed738c
2019-10-01T00:00:00Z,view,3900821,2053013552326770905,appliances.environment.water_heater,aqua,33.20,554748717,9333dfbd-b87a-4708-9857-6336556b0fcc
2019-10-01T00:00:01Z,view,17200506,2053013559792632471,furniture.living_room.sofa,null,543.10,519107250,566511c2-e2e3-422b-b695-cf8e6e792ca8
2019-10-01T00:00:01Z,view,1307067,2053013558920217191,computers.notebook,lenovo,251.74,550050854,7c90fc70-0e80-4590-96f3-13c02c18c713
2019-10-01T00:00:04Z,view,1004237,2053013555631882655,electronics.smartphone,apple,1081.98,535871217,c6bd7419-2748-4c56-95b4-8cec9ff8b80d


## Task 11: Flag the bad records

In [0]:
valid_types = ["view", "cart", "remove_from_cart", "purchase"]

flagged = typed.withColumn(
    "reject_reason",
    F.when(F.col("event_time").isNull(), "bad_event_time")
     .when(~F.col("event_type").isin(valid_types) | F.col("event_type").isNull(), "bad_event_type")
     .when(F.col("product_id").isNull(), "missing_product_id")
     .when(F.col("user_id").isNull(), "missing_user_id")
     .when(F.col("price").isNull() | (F.col("price") <= 0), "bad_price"),
)

rejected = flagged.filter(F.col("reject_reason").isNotNull())
good = flagged.filter(F.col("reject_reason").isNull()).drop("reject_reason")

## Task 12: Clean the good rows

In [0]:
clean = (
    good
    .fillna({"brand": "unknown", "category_code": "unknown"})
    .withColumn("main_category", F.split("category_code", r"\.").getItem(0))
    .withColumn("event_date", F.to_date("event_time"))
    .dropDuplicates(["event_time", "event_type", "product_id", "user_id", "user_session"])
)

## Task 13: Write the rejected rows

In [0]:
(rejected
    .write.format("delta")
    .mode("overwrite")
    .save(rejected_path))

## Task 14: Write silver

In [0]:
(clean
    .repartition("event_date")
    .write.format("delta")
    .mode("overwrite")
    .partitionBy("event_date")
    .save(silver_path))

## Task 15: Look at what you built

In [0]:
display(dbutils.fs.ls(silver_path))

events = spark.read.format("delta").load(silver_path)
events.printSchema()
display(events.limit(10))

path,name,size,modificationTime
abfss://silver@stbigdatadev00001.dfs.core.windows.net/ecommerce_events/_delta_log/,_delta_log/,0,1793868932000
abfss://silver@stbigdatadev00001.dfs.core.windows.net/ecommerce_events/event_date=2019-10-01/,event_date=2019-10-01/,0,1793869667000
abfss://silver@stbigdatadev00001.dfs.core.windows.net/ecommerce_events/event_date=2019-10-02/,event_date=2019-10-02/,0,1793869631000
abfss://silver@stbigdatadev00001.dfs.core.windows.net/ecommerce_events/event_date=2019-10-03/,event_date=2019-10-03/,0,1793869744000
abfss://silver@stbigdatadev00001.dfs.core.windows.net/ecommerce_events/event_date=2019-10-04/,event_date=2019-10-04/,0,1793869727000
abfss://silver@stbigdatadev00001.dfs.core.windows.net/ecommerce_events/event_date=2019-10-05/,event_date=2019-10-05/,0,1793869733000
abfss://silver@stbigdatadev00001.dfs.core.windows.net/ecommerce_events/event_date=2019-10-06/,event_date=2019-10-06/,0,1793869676000
abfss://silver@stbigdatadev00001.dfs.core.windows.net/ecommerce_events/event_date=2019-10-07/,event_date=2019-10-07/,0,1793869684000
abfss://silver@stbigdatadev00001.dfs.core.windows.net/ecommerce_events/event_date=2019-10-08/,event_date=2019-10-08/,0,1793869732000
abfss://silver@stbigdatadev00001.dfs.core.windows.net/ecommerce_events/event_date=2019-10-09/,event_date=2019-10-09/,0,1793869738000


root
 |-- event_time: timestamp (nullable = true)
 |-- event_type: string (nullable = true)
 |-- product_id: long (nullable = true)
 |-- category_id: long (nullable = true)
 |-- category_code: string (nullable = true)
 |-- brand: string (nullable = true)
 |-- price: decimal(10,2) (nullable = true)
 |-- user_id: long (nullable = true)
 |-- user_session: string (nullable = true)
 |-- main_category: string (nullable = true)
 |-- event_date: date (nullable = true)



event_time,event_type,product_id,category_id,category_code,brand,price,user_id,user_session,main_category,event_date
2019-11-01T00:02:32Z,view,58900001,2159649186167915110,unknown,total,15.06,548804259,d8a4a956-1d7a-4ae8-95f1-605402901cef,unknown,2019-11-01
2019-11-01T00:02:35Z,view,16900119,2053013558433677895,furniture.kitchen.table,unknown,57.38,548523725,9f4e068b-d942-40c1-879f-e0de101e6157,furniture,2019-11-01
2019-11-01T00:04:07Z,view,1801766,2053013554415534427,electronics.video.tv,artel,146.85,543653226,ab310b47-1eb2-45f8-8e5b-21ab2010925a,electronics,2019-11-01
2019-11-01T00:05:32Z,view,1003910,2053013555631882655,electronics.smartphone,nokia,118.10,528269305,6dce75d2-e455-4e52-ab65-af16e36f64cb,electronics,2019-11-01
2019-11-01T00:09:16Z,view,1002544,2053013555631882655,electronics.smartphone,apple,458.28,540508491,33a0d8f4-c388-4ce7-85f4-7b1f11563d6a,electronics,2019-11-01
2019-11-01T00:10:24Z,view,4100126,2053013561218695907,unknown,sony,331.01,516426931,ef4867dd-b922-4d92-abec-9a75acb2b769,unknown,2019-11-01
2019-11-01T00:10:34Z,view,1005265,2053013555631882655,electronics.smartphone,xiaomi,203.27,513198013,5f5e7618-afc5-41c9-abaf-dde70dd8dd50,electronics,2019-11-01
2019-11-01T00:12:24Z,view,1801766,2053013554415534427,electronics.video.tv,artel,146.85,515782589,f2081cf0-0ee3-4bf1-9b40-e5e580f9ace3,electronics,2019-11-01
2019-11-01T00:15:23Z,view,1004870,2053013555631882655,electronics.smartphone,samsung,275.25,513241713,8ce43266-bfa2-4f1a-9e16-8f2f8fb4b6aa,electronics,2019-11-01
2019-11-01T00:17:47Z,view,13200944,2053013557192163841,furniture.bedroom.bed,sv,282.89,560134869,a48a30b6-61b5-4816-8fe6-d9e69b65aed4,furniture,2019-11-01


### Look at the table history, which comes from the Delta transaction log:

In [0]:
display(spark.sql(f"DESCRIBE HISTORY delta.`{silver_path}`"))

version,timestamp,userId,userName,operation,operationParameters,job,notebook,queryHistoryStatementId,clusterId,readVersion,isolationLevel,isBlindAppend,operationMetrics,userMetadata,engineInfo
0,2026-11-05T09:09:11Z,149290047024048,serhiy.dataskools@gmail.com,WRITE,"Map(mode -> Overwrite, statsOnLoad -> false, partitionBy -> [""event_date""])",null,List(593119271945842),null,1104-140239-5vazfb1e,null,WriteSerializable,false,"Map(numFiles -> 61, numRemovedFiles -> 0, numRemovedBytes -> 0, numDeletionVectorsRemoved -> 0, numOutputRows -> 109563336, numOutputBytes -> 5969717136)",null,Databricks-Runtime/18.x-photon-scala2.13


## END


# ========================      Data quality checks      ========================

## Part E - Data Quality Checks

## Task 16: Reconcile the counts

In [0]:
raw_count = raw.count()
rejected_df = spark.read.format("delta").load(rejected_path)
rejected_count = rejected_df.count()
silver_count = events.count()
dupes_removed = raw_count - rejected_count - silver_count

print(f"raw rows:           {raw_count:,}")
print(f"rejected rows:      {rejected_count:,}")
print(f"silver rows:        {silver_count:,}")
print(f"duplicates removed: {dupes_removed:,}")

raw rows:           109,950,743
rejected rows:      256,761
silver rows:        109,563,336
duplicates removed: 130,646


## Task 17: Why were rows rejected?

In [0]:
display(rejected_df.groupBy("reject_reason").count().orderBy(F.desc("count")))

reject_reason,count
bad_price,256761


## Task 18: Check the silver table itself
#### Nulls in every column

In [0]:
display(events.select([F.sum(F.col(c).isNull().cast("int")).alias(c) for c in events.columns]))

event_time,event_type,product_id,category_id,category_code,brand,price,user_id,user_session,main_category,event_date
0,0,0,0,0,0,0,0,12,0,0


#### Valid values and price range

In [0]:
display(events.groupBy("event_type").count())
display(events.agg(F.min("price"), F.max("price"), F.min("event_date"), F.max("event_date")))

event_type,count
purchase,1659703
view,104079641
cart,3823992


min(price),max(price),min(event_date),max(event_date)
0.77,2574.07,2019-10-01,2019-11-30


#### Any duplicates left (the answer must be 0):

In [0]:
key = ["event_time", "event_type", "product_id", "user_id", "user_session"]
print(events.groupBy(key).count().filter("count > 1").count())

0


END